# 2. Advanced Threat Hunting

## Hypothesis-driven hunting

Professional threat hunting follows a cycle:

1. **Hypothesis**: "I think an attacker used compromised credentials to access our database server."
2. **Query**: Write KQL/API queries to find evidence.
3. **Analyze**: Do the results support or refute the hypothesis?
4. **Action**: If confirmed → create detection rule + incident. If not → refine hypothesis.

Let's practice three hunting hypotheses against our SIEM data.

In [ ]:
import httpx, json
from collections import Counter, defaultdict

SIEM = 'http://localhost:8000'

def query(table, filter=None, aggregate_by=None, limit=500):
    r = httpx.post(f'{SIEM}/query', json={'table_name': table, 'filter': filter, 'aggregate_by': aggregate_by, 'limit': limit})
    return r.json()['results']

# ===== HYPOTHESIS 1: Account compromise via brute force =====
print('╔══════════════════════════════════════════════════════════╗')
print('║ HYPOTHESIS 1: An account was compromised via brute force║')
print('╚══════════════════════════════════════════════════════════╝\n')

# Find users with high failure rates
all_signins = query('SigninLogs', limit=500)
user_signin = defaultdict(lambda: {'fail': 0, 'success': 0, 'fail_ips': set(), 'success_ips': set(), 'locs': set()})
for s in all_signins:
    u = s['UserPrincipalName']
    if s['ResultType'] == 'Failure':
        user_signin[u]['fail'] += 1
        user_signin[u]['fail_ips'].add(s['IPAddress'])
    else:
        user_signin[u]['success'] += 1
        user_signin[u]['success_ips'].add(s['IPAddress'])
    user_signin[u]['locs'].add(s['Location'])

compromised = [(u, d) for u, d in user_signin.items() if d['fail'] > 5 and d['success'] > 0]
if compromised:
    print('✅ HYPOTHESIS CONFIRMED\n')
    for user, data in compromised:
        print(f'  Account: {user}')
        print(f'  Failed attempts: {data["fail"]} from IPs: {data["fail_ips"]}')
        print(f'  Successful logins: {data["success"]} from IPs: {data["success_ips"]}')
        print(f'  Locations: {data["locs"]}')
        overlap = data['fail_ips'] & data['success_ips']
        if overlap:
            print(f'  ⚠️  SAME IP used for failures AND success: {overlap}')
            print(f'  → Attacker succeeded after brute forcing!')
else:
    print('❌ HYPOTHESIS NOT CONFIRMED — no brute force patterns found.')

In [ ]:
# ===== HYPOTHESIS 2: Compromised account performed lateral movement =====
print('╔══════════════════════════════════════════════════════════════╗')
print('║ HYPOTHESIS 2: Compromised account moved laterally to servers║')
print('╚══════════════════════════════════════════════════════════════╝\n')

# Get the compromised user from H1
if compromised:
    target_user = compromised[0][0].split('@')[0]  # e.g. 'alice'
    
    # Check endpoint activity for this user
    endpoint_events = query('DeviceEvents', filter={'AccountName': target_user})
    
    # Analyze: which devices, which processes
    devices = Counter(e['DeviceName'] for e in endpoint_events)
    processes = Counter(e['FileName'] for e in endpoint_events)
    remote_exec = [e for e in endpoint_events if e['ActionType'] == 'RemoteExecution']
    attack_tools = [e for e in endpoint_events if e['FileName'] in ('mimikatz.exe', 'psexec.exe')]
    
    print(f'User: {target_user}')
    print(f'Devices touched: {dict(devices)}')
    print(f'Processes run: {dict(processes)}')
    print(f'Remote executions: {len(remote_exec)}')
    print(f'Attack tools used: {len(attack_tools)}')
    
    if len(devices) > 1 and remote_exec:
        print(f'\n✅ HYPOTHESIS CONFIRMED')
        print(f'  {target_user} accessed {len(devices)} devices with remote execution.')
        print(f'  Attack tools detected: {", ".join(e["FileName"] for e in attack_tools)}')
        print(f'\n  → ACTION: Isolate all affected devices, disable user, investigate data access.')
    else:
        print('\n❌ HYPOTHESIS NOT CONFIRMED')

In [ ]:
# ===== HYPOTHESIS 3: Data was exfiltrated to external destinations =====
print('╔═══════════════════════════════════════════════════════════╗')
print('║ HYPOTHESIS 3: Data was exfiltrated to external endpoints ║')
print('╚═══════════════════════════════════════════════════════════╝\n')

# Check firewall for outbound connections to non-internal IPs
fw_events = query('AzureFirewall', limit=500)
external = [e for e in fw_events if not e.get('DestinationIP', '').startswith('10.')]
external_ips = Counter(e['DestinationIP'] for e in external)

known_bad_ips = {'185.220.101.42', '45.33.32.156', '198.51.100.99'}

print(f'Total firewall events: {len(fw_events)}')
print(f'External connections: {len(external)}')
print(f'\nExternal destination IPs:')

exfil_confirmed = False
for ip, count in external_ips.most_common():
    threat = '🔴 KNOWN C2/EXFIL' if ip in known_bad_ips else '⬜'
    if ip in known_bad_ips:
        exfil_confirmed = True
    print(f'  {ip:<20} {count:>3} connections  {threat}')

if exfil_confirmed:
    # Trace which internal host connected to bad IPs
    for bad_ip in known_bad_ips:
        sources = [e['SourceIP'] for e in external if e['DestinationIP'] == bad_ip]
        if sources:
            print(f'\n  {bad_ip} was contacted by: {Counter(sources).most_common()}')
    
    # Cross-reference with endpoint events
    upload_events = query('DeviceEvents', filter={'ActionType': 'FileUploaded'})
    if upload_events:
        print(f'\n  File upload events: {len(upload_events)}')
        devices = Counter(e['DeviceName'] for e in upload_events)
        print(f'  Uploading devices: {dict(devices)}')
    
    print(f'\n✅ HYPOTHESIS CONFIRMED — data exfiltration detected!')
    print(f'  → ACTION: Block IPs in firewall, isolate source hosts, assess data loss.')
else:
    print('\n❌ HYPOTHESIS NOT CONFIRMED')

## Creating a detection rule from a hunt

When your hunt finds something real, **convert it to an analytics rule** so it's detected automatically next time.

In [ ]:
# Convert our exfiltration hunt into a permanent detection rule
print('=== Converting hunt to detection rule ===\n')

new_rule = {
    'name': 'High-volume outbound to external IP',
    'severity': 'High',
    'tactic': 'Exfiltration',
    'query_table': 'AzureFirewall',
    'aggregate_by': 'DestinationIP',
    'threshold': 5,
    'window_minutes': 60,
    'description': 'Detects >5 outbound connections to a single external IP in 1 hour. May indicate data exfiltration.',
}

r = httpx.post(f'{SIEM}/rules', json=new_rule)
print(f'Rule created: {r.json()}')
print(f'\nReal KQL equivalent:')
print(f'''  AzureFirewall
  | where TimeGenerated > ago(1h)
  | where DestinationIP !startswith "10."
  | summarize ConnectionCount=count() by DestinationIP, SourceIP
  | where ConnectionCount > 5''')

print(f'\n💡 This is the threat hunting cycle:')
print(f'   Hypothesis → Query → Confirm → Create Rule → Automate')

## SC-200 hunting exam tips

### Sentinel-specific hunting features

| Feature | What it does |
|---------|-------------|
| **Hunting queries** | Pre-built KQL queries organized by MITRE tactic |
| **Bookmarks** | Save interesting query results for later investigation |
| **Livestream** | Real-time query results as events flow in |
| **Notebooks** | Jupyter notebooks connected to Sentinel data |
| **Hunting graph** | Visual entity relationships |
| **Data lake tier** | Low-cost storage for long-term hunting data |

### Advanced Hunting in Defender XDR

| Feature | What it does |
|---------|-------------|
| **Custom detection rules** | Save a hunting query as a scheduled detection |
| **Shared queries** | Share with team |
| **Blast radius** | Graph showing how far an attack could spread |
| **Go hunt** | One-click hunting from an incident or entity |

### The hunting cycle

```
    ┌──────────────┐
    │  Threat Intel │ ← external IOCs, MITRE techniques
    └──────┬───────┘
           ▼
    ┌──────────────┐
    │  Hypothesis  │ ← "attacker may be doing X"
    └──────┬───────┘
           ▼
    ┌──────────────┐
    │  Query Data  │ ← KQL across relevant tables
    └──────┬───────┘
           ▼
    ┌──────────────┐
    │  Analyze     │ ← confirm / refute
    └──────┬───────┘
           ▼
    ┌──────────────┐
    │  Action      │ ← create rule, incident, or refine hypothesis
    └──────┬───────┘
           │
           └──────────▶ repeat
```

---
## You've completed all SC-200 labs!

### What you built and practiced

1. A **working mini-SIEM** with log ingestion, query engine, analytics rules, and playbooks
2. **Multi-stage attack investigation** across 4 data sources
3. **Incident response workflows** — triage, investigate, contain, remediate, close
4. **Threat hunting** with hypothesis-driven queries and rule creation

### Next steps

1. Take the [SC-200 practice assessment](https://learn.microsoft.com/en-us/credentials/certifications/security-operations-analyst/practice/assessment?assessment-type=practice&assessmentId=59&practice-assessment-type=certification)
2. Practice KQL at [detective.kusto.io](https://detective.kusto.io)
3. Review the Microsoft Learn paths listed in the main README